In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, \
    matthews_corrcoef, recall_score, confusion_matrix

from torch.utils.tensorboard import SummaryWriter

import pandas as pd

import numpy as np

import os

from pathlib import Path

import seaborn as sns

from datetime import datetime as dt

import joblib

import nltk
from nltk.corpus import stopwords

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [7]:
script_dir = Path().cwd()

models_folder = os.path.join(script_dir, r'results\LinearSVM\models')
os.makedirs(models_folder, exist_ok=True)

results_folder = os.path.join(script_dir, r'results')

In [8]:
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

scaler = StandardScaler()

stop_words = stopwords.words('dutch')

vectorizer = TfidfVectorizer(
    stop_words = stopwords.words('dutch'),
    max_df = 0.95,
    min_df = 0.05
)


In [9]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

labels = df.iloc[:, -1]
df = df.iloc[:, 1:-1]

In [10]:
indices = np.arange(df.shape[0])

trainval_indices, test_indices = train_test_split(
    indices,
    test_size = TEST_SIZE, 
    stratify = labels, 
    random_state = SEED
)

df_test = df.iloc[test_indices]
y_test = labels.iloc[test_indices]

df_trainval = df.iloc[trainval_indices]
y_trainval = labels.iloc[trainval_indices]

In [11]:
# Stratified KFold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Inizializzazione
scaler = StandardScaler()
vectorizer = TfidfVectorizer()

for split, (train_indices, val_indices) in enumerate(skf.split(df_trainval, y_trainval)):
    model = LinearSVC(random_state=SEED)
    
    df_train = df_trainval.iloc[train_indices]
    y_train = y_trainval.iloc[train_indices]
    
    X_texts = vectorizer.fit_transform(df_train['consult']).toarray()
    X_data = df_train.drop(columns='consult').values
    
    X_train = np.concatenate((X_texts, X_data), axis=1)
    
    X_train = scaler.fit_transform(X_train)
    
    # Addestra il modello
    model.fit(X_train, y_train)
    
    joblib.dump({
            'model': model,
            'scaler': scaler,
            'vectorizer': vectorizer
        },
        os.path.join(models_folder, f'split_{split}')
    )

In [20]:
writer_folder = os.path.join(results_folder, f'LinearSVM\\{dt.now().strftime("%Y_%m_%d_%H_%M")}')

skf = StratifiedKFold(
    n_splits = N_SPLITS,
    shuffle = True,
    random_state = SEED
)

test_metrics = []

for split, filename in enumerate(os.listdir(models_folder)):
    
    writer_path = os.path.join(writer_folder, f'split_{split}')
    writer = SummaryWriter(log_dir = writer_path)
    
    loaded_dict = joblib.load(os.path.join(models_folder, filename))
    
    model, scaler, vectorizer= loaded_dict['model'], loaded_dict['scaler'], \
        loaded_dict['vectorizer']
    
    df_val = df_trainval.iloc[val_indices]
    y_val = y_trainval.iloc[val_indices]
    
    X_texts = vectorizer.transform(df_val['consult']).toarray()
    X_data = df_val.drop(columns='consult').values
    
    X_val = np.concatenate((X_texts, X_data), axis=1)
    
    X_val = scaler.transform(X_val)
    
    val_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred, average='macro')  # Puoi cambiare 'macro' con 'micro', 'weighted', o 'binary' se necessario
    prc = precision_score(y_val, val_pred, average='macro')  # Analogamente, scegli l'average adatto
    mcc = matthews_corrcoef(y_val, val_pred)
    rec = recall_score(y_val, val_pred, average='macro')
    cm = confusion_matrix(y_val, val_pred)
    
    # Log to TensorBoard
    writer.add_scalar('Accuracy/val',  acc)
    writer.add_scalar('F1_Score/val',  f1)
    writer.add_scalar('Precision/val', prc)
    writer.add_scalar('MCC_Score/val', mcc)
    writer.add_scalar('Recall/val',    rec)
    writer.add_figure('Confusion_matrix/val', sns.heatmap(cm, annot=True).get_figure())
    
    X_texts = vectorizer.transform(df_test['consult']).toarray()
    X_data = df_test.drop(columns='consult').values
    
    X_test = np.concatenate((X_texts, X_data), axis=1)
    
    X_test = scaler.transform(X_test)
    
    test_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred, average='macro')  # Puoi cambiare 'macro' con 'micro', 'weighted', o 'binary' se necessario
    prc = precision_score(y_test, test_pred, average='macro')  # Analogamente, scegli l'average adatto
    mcc = matthews_corrcoef(y_test, test_pred)
    rec = recall_score(y_test, test_pred, average='macro')
    cm = confusion_matrix(y_test, test_pred)
    
    test_metrics.append([acc*100, f1*100, prc*100, mcc, rec*100])
    
    # Log to TensorBoard
    writer.add_scalar('Accuracy/test',  acc)
    writer.add_scalar('F1_Score/test',  f1)
    writer.add_scalar('Precision/test', prc)
    writer.add_scalar('MCC_Score/test', mcc)
    writer.add_scalar('Recall/test',    rec)
    writer.add_figure('Confusion_matrix/test', sns.heatmap(cm, annot=True).get_figure())

test_metrics = np.array(test_metrics)

test_avg = test_metrics.mean(axis=0)
test_std = np.std(test_metrics, axis=0)

performance_path = os.path.join(writer_folder, 'performance.txt')
with open(performance_path, 'w') as file:
    file.write(f"Average Test Metrics\n")
    file.write(f"- Accuracy: {test_avg[0]:.2f} ({test_std[0]:.2f})\n")
    file.write(f"- F1: {test_avg[1]:.2f} ({test_std[1]:.2f})\n")
    file.write(f"- Precision: {test_avg[2]:.2f} ({test_std[2]:.2f})\n")
    file.write(f"- MCC: {test_avg[3]:.3f} ({test_std[3]:.3f})\n")
    file.write(f"- Recall: {test_avg[4]:.2f} ({test_std[4]:.2f})")

In [ ]:
np.std(test_metrics[1], axis=0)

# I need to find the most important words for the baseline model.
## Try using SHAP or LIME

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, \
    matthews_corrcoef, recall_score, confusion_matrix

import pandas as pd

import numpy as np

import joblib

import shap

from pathlib import Path

import os

import nltk
from nltk.corpus import stopwords

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [3]:
script_dir = Path.cwd()

models_folder = os.path.join(script_dir, r'results\SVM\models')
os.makedirs(models_folder, exist_ok=True)

In [4]:
SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

scaler = StandardScaler()

stop_words = stopwords.words('dutch')

vectorizer = TfidfVectorizer(
    stop_words = stopwords.words('dutch'),
    max_df = 0.95,
    min_df = 0.05
)

In [5]:
df = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_consults_medication.parquet')

labels = df.iloc[:, -1]
df = df.iloc[:, 1:-1]

In [6]:
indices = np.arange(df.shape[0])

trainval_indices, test_indices = train_test_split(
    indices,
    test_size = TEST_SIZE, 
    stratify = labels, 
    random_state = SEED
)

df_test = df.iloc[test_indices]
y_test = labels.iloc[test_indices]

df_trainval = df.iloc[trainval_indices]
labels_trainval = labels.iloc[trainval_indices]


In [7]:
# Stratified KFold
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Inizializzazione
scaler = StandardScaler()
vectorizer = TfidfVectorizer()

for split, (train_indices, val_indices) in enumerate(skf.split(df_trainval, labels_trainval)):
    model = LinearSVC(random_state=SEED)
    
    df_train = df_trainval.iloc[train_indices]
    y_train = labels_trainval.iloc[train_indices]
    
    X_train = vectorizer.fit_transform(df_train['consult']).toarray()
    
    X_train = scaler.fit_transform(X_train)
    
    # Addestra il modello
    model.fit(X_train, y_train)
    
    explainer = shap.LinearExplainer(model, X_train)
    
    joblib.dump({
            'model': model,
            'scaler': scaler,
            'vectorizer': vectorizer,
            'explainer': explainer
        },
        os.path.join(models_folder, f'split_{split}')
    )

In [ ]:
for filename in os.listdir(models_folder):
    path = os.path.join(models_folder, filename)
    loaded_dict = joblib.load(path)
    
    model, scaler, vectorizer, explainer = loaded_dict['model'], loaded_dict['scaler'], \
        loaded_dict['vectorizer'], loaded_dict['explainer']
    
    df_val = df_trainval.iloc[val_indices]
    y_val = labels_trainval.iloc[val_indices]
    
    X_val = vectorizer.transform(df_val['consult']).toarray()
    
    X_val = scaler.transform(X_val)
    
    # Calcola le predizioni sul set di validazione
    val_pred = model.predict(X_val)
    
    # Calcolo delle metriche
    acc = accuracy_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred, average='macro')
    prc = precision_score(y_val, val_pred, average='macro')
    mcc = matthews_corrcoef(y_val, val_pred)
    rec = recall_score(y_val, val_pred, average='macro')
    cm = confusion_matrix(y_val, val_pred)
    
    print(f'Validation\nAccuracy: {acc:.4f},\tF1: {f1:.4f},\tPrecision: {prc:.4f},\tRecall: {rec:.4f},\tMCC: {mcc:.4f}')
    
    # Calcolare i valori SHAP per il test set
    shap_values = explainer.shap_values(X_val)

    # Ordina le feature per importanza (sommando i valori assoluti dei SHAP values per ogni feature)
    shap_importance = np.abs(shap_values).mean(axis=0)  # Media dei valori assoluti per feature
    sorted_idx = np.argsort(shap_importance)[::-1]  # Ordina dalla feature più importante alla meno importante

    # Seleziona solo le parole più significative per il violin plot (max_display definisce quante parole visualizzare)
    max_display = 20
    top_features_idx = sorted_idx[:max_display]

    # Visualizza il violin plot per le parole più significative
    shap.violin_plot(shap_values[:, top_features_idx], X_val[:, top_features_idx], feature_names=vectorizer.get_feature_names_out()[top_features_idx])
    
    # Esegui la stessa procedura per il test set
    X_test = vectorizer.transform(df_test['consult']).toarray()
    
    X_test = scaler.transform(X_test)
    
    test_pred = model.predict(X_test)
    
    # Calcola le metriche per il test set
    acc = accuracy_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred, average='macro')
    prc = precision_score(y_test, test_pred, average='macro')
    mcc = matthews_corrcoef(y_test, test_pred)
    rec = recall_score(y_test, test_pred, average='macro')
    cm = confusion_matrix(y_test, test_pred)
    
    print(f'\nTest\nAccuracy: {acc:.4f},\tF1: {f1:.4f},\tPrecision: {prc:.4f},\tRecall: {rec:.4f},\tMCC: {mcc:.4f}\n')
    
    # Calcolare i valori SHAP per il test set
    shap_values = explainer.shap_values(X_test)

    # Ordina le feature per importanza (sommando i valori assoluti dei SHAP values per ogni feature)
    shap_importance = np.abs(shap_values).mean(axis=0)  # Media dei valori assoluti per feature
    sorted_idx = np.argsort(shap_importance)[::-1]  # Ordina dalla feature più importante alla meno importante

    # Seleziona solo le parole più significative per il violin plot (max_display definisce quante parole visualizzare)
    max_display = 20
    top_features_idx = sorted_idx[:max_display]

    # Visualizza il violin plot per le parole più significative
    shap.violin_plot(shap_values[:, top_features_idx], X_test[:, top_features_idx], feature_names=vectorizer.get_feature_names_out()[top_features_idx])
